In [ ]:
!pip install -q huggingface_hub datasets monai scikit-learn pandas matplotlib seaborn

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from datasets import load_dataset
from torch.utils.data import Dataset as TorchDataset, DataLoader
from monai.transforms import Compose, ScaleIntensityd, Resized, ToTensord
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# تحميل بيانات الاختبار فقط
print("جاري تحميل بيانات الاختبار...")
dataset = load_dataset("hf-vision/chest-xray-pneumonia", split='test')

class HFToMONAIDataset(TorchDataset):
    def __init__(self, hf_dataset, transforms=None):
        self.hf_dataset = hf_dataset
        self.transforms = transforms

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        img = np.array(item['image'].convert('RGB'), dtype=np.float32)
        img = np.transpose(img, (2, 0, 1)) 
        data_dict = {"image": img, "label": np.array(item['label'], dtype=np.int64)}
        if self.transforms:
            data_dict = self.transforms(data_dict)
        return data_dict

test_transforms = Compose([
    ScaleIntensityd(keys=["image"]),
    Resized(keys=["image"], spatial_size=(224, 224)),
    ToTensord(keys=["image", "label"])
])

test_ds = HFToMONAIDataset(dataset, transforms=test_transforms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
print(f"تم تجهيز بيانات الاختبار. عدد الدفعات: {len(test_loader)}")

In [ ]:
repo_id = "talalsvu/chest-xray-base-models"
num_classes = 2

print("جاري تحميل الأوزان من Hugging Face...")
resnet_path = hf_hub_download(repo_id=repo_id, filename="resnet50_best_weights.pth")
densenet_path = hf_hub_download(repo_id=repo_id, filename="densenet121_best_weights.pth")
efficientnet_path = hf_hub_download(repo_id=repo_id, filename="efficientnet_b0_best_weights.pth")

# 1. ResNet-50
resnet50 = models.resnet50(weights=None)
resnet50.fc = nn.Linear(resnet50.fc.in_features, num_classes)
resnet50.load_state_dict(torch.load(resnet_path, map_location=device))
resnet50 = resnet50.to(device).eval()

# 2. DenseNet-121
densenet121 = models.densenet121(weights=None)
densenet121.classifier = nn.Linear(densenet121.classifier.in_features, num_classes)
densenet121.load_state_dict(torch.load(densenet_path, map_location=device))
densenet121 = densenet121.to(device).eval()

# 3. EfficientNet-B0
efficientnet_b0 = models.efficientnet_b0(weights=None)
efficientnet_b0.classifier[1] = nn.Linear(efficientnet_b0.classifier[1].in_features, num_classes)
efficientnet_b0.load_state_dict(torch.load(efficientnet_path, map_location=device))
efficientnet_b0 = efficientnet_b0.to(device).eval()

print("تم بناء النماذج الثلاثة وتحميل أوزانها الجاهزة بنجاح!")

In [ ]:
class EntropyDynamicEnsemble(nn.Module):
    """
    نظام هجين تكيفي (Dynamic Ensemble) يعتمد على الإنتروبيا (Entropy).
    يقوم بحساب الشك (Uncertainty) لكل نموذج لكل صورة بشكل مستقل.
    """
    def __init__(self, model1, model2, model3):
        super(EntropyDynamicEnsemble, self).__init__()
        self.model1 = model1
        self.model2 = model2
        self.model3 = model3
        
        # تجميد الأوزان 
        for param in self.parameters():
            param.requires_grad = False

    def compute_entropy(self, probs):
        return -torch.sum(probs * torch.log(probs + 1e-6), dim=1)

    def forward(self, x):
        probs1 = F.softmax(self.model1(x), dim=1)
        probs2 = F.softmax(self.model2(x), dim=1)
        probs3 = F.softmax(self.model3(x), dim=1)

        ent1 = self.compute_entropy(probs1)
        ent2 = self.compute_entropy(probs2)
        ent3 = self.compute_entropy(probs3)

        entropies = torch.stack([ent1, ent2, ent3], dim=1)
        weights = F.softmax(-entropies, dim=1)

        w1 = weights[:, 0].unsqueeze(1)
        w2 = weights[:, 1].unsqueeze(1)
        w3 = weights[:, 2].unsqueeze(1)

        dynamic_probs = (w1 * probs1) + (w2 * probs2) + (w3 * probs3)
        return dynamic_probs

# بناء النموذج الهجين
hybrid_model = EntropyDynamicEnsemble(resnet50, densenet121, efficientnet_b0).to(device)
hybrid_model.eval()

print("تم بناء خوارزمية النظام الهجين التكيفي (Entropy-Based) بنجاح!")

In [ ]:
true_labels = []
ensemble_pred_labels = []
ensemble_pred_probs = []

print("جاري تقييم النظام الهجين (Dynamic Ensemble) على بيانات الاختبار...")
with torch.no_grad():
    for batch in test_loader:
        inputs = batch["image"].to(device)
        labels = batch["label"].to(device)
        
        # تمرير الصور مباشرة للنظام الهجين
        final_probs = hybrid_model(inputs)
        _, preds = torch.max(final_probs, 1)
        
        true_labels.extend(labels.cpu().numpy())
        ensemble_pred_labels.extend(preds.cpu().numpy())
        ensemble_pred_probs.extend(final_probs[:, 1].cpu().numpy()) # احتمال فئة Pneumonia

target_names = ['Normal (0)', 'Pneumonia (1)']
print("\n--- تقرير تصنيف النظام الهجين الديناميكي (Entropy-Based) ---")
print(classification_report(true_labels, ensemble_pred_labels, target_names=target_names))

roc_auc = roc_auc_score(true_labels, ensemble_pred_probs)
print(f"ROC-AUC Score: {roc_auc:.4f}\n")

cm = confusion_matrix(true_labels, ensemble_pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - Dynamic Ensemble (Chest X-Ray)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
print("جاري تقييم النظام باستخدام التجميع الثابت (Average Ensembling) للمقارنة...")

static_pred_labels = []
static_pred_probs = []
true_labels_list = []

with torch.no_grad():
    for batch in test_loader: 
        inputs = batch["image"].to(device)
        labels = batch["label"].to(device)
        
        probs1 = torch.softmax(resnet50(inputs), dim=1)
        probs2 = torch.softmax(densenet121(inputs), dim=1)
        probs3 = torch.softmax(efficientnet_b0(inputs), dim=1)
        
        # التجميع الثابت
        avg_probs = (probs1 + probs2 + probs3) / 3.0
        
        _, preds = torch.max(avg_probs, 1)
        
        true_labels_list.extend(labels.cpu().numpy())
        static_pred_labels.extend(preds.cpu().numpy())
        static_pred_probs.extend(avg_probs[:, 1].cpu().numpy()) 

target_names_xray = ['Normal', 'Pneumonia']

print("\n--- تقرير تصنيف التجميع الثابت (Static Average Ensemble) - Chest X-Ray ---")
print(classification_report(true_labels_list, static_pred_labels, target_names=target_names_xray))

roc_auc_static = roc_auc_score(true_labels_list, static_pred_probs)
print(f"Static Ensemble ROC-AUC Score: {roc_auc_static:.4f}\n")

cm_static = confusion_matrix(true_labels_list, static_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_static, annot=True, fmt='d', cmap='Oranges', xticklabels=target_names_xray, yticklabels=target_names_xray)
plt.title('Confusion Matrix - Static Ensemble (Chest X-Ray)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
!pip install -q grad-cam

import cv2
import numpy as np
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print("جاري توليد الخريطة الحرارية (Grad-CAM) لقابلية التفسير...")

# 1. جلب صورة واحدة من بيانات الاختبار للأشعة السينية
model_explain = resnet50.eval()

# === تفعيل التدرجات لنموذج التفسير ===
for param in model_explain.parameters():
    param.requires_grad = True
# ==================================

sample_batch = next(iter(test_loader))
input_tensor = sample_batch["image"][0].unsqueeze(0).to(device)
true_label = sample_batch["label"][0].item()

# 2. تحضير الصورة للعرض
img_show = input_tensor.squeeze().cpu().numpy()
img_show = np.transpose(img_show, (1, 2, 0)) 
img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min()) 

# 3. إعداد Grad-CAM
target_layers = [model_explain.layer4[-1]]
cam = GradCAM(model=model_explain, target_layers=target_layers)

targets = [ClassifierOutputTarget(true_label)]

# 4. توليد الخريطة الحرارية
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
grayscale_cam = grayscale_cam[0, :]

visualization = show_cam_on_image(img_show, grayscale_cam, use_rgb=True)

# 5. رسم النتيجة
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_show)
axes[0].set_title(f"Original Chest X-Ray\nTrue Label: {target_names_xray[true_label]}")
axes[0].axis('off')

axes[1].imshow(visualization)
axes[1].set_title("Grad-CAM Heatmap\n(Where the model looked)")
axes[1].axis('off')

plt.tight_layout()
plt.show()